# Instrumental Variables (IV)

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


**Business setting:** treatment is self-selected and confounded by **unobservables**, but some variable shifts treatment exposure without otherwise affecting the outcome. Classic business instruments:

- **Encouragement designs:** a randomly sent email nudging feature adoption — randomized encouragement, self-selected adoption. (This is the cleanest business IV because you control the instrument.)
- **Supply shocks:** regional shipping delays as an instrument for delivery speed's effect on repeat purchase.
- **Distance / availability:** distance to the nearest store as an instrument for in-store program exposure.
- **Quasi-random assignment of agents:** customers quasi-randomly routed to support reps with different propensities to offer a discount ("judge leniency" design).

**Conditions for a valid instrument Z:**

1. **Relevance:** Z moves D (testable — first-stage F-statistic).
2. **Exclusion:** Z affects Y *only through* D (untestable; argue from institutional knowledge).
3. **Independence:** Z is as good as random w.r.t. potential outcomes.
4. **Monotonicity:** no defiers (no one who adopts *because* they were *not* encouraged).

Under these, IV identifies the **LATE** — the effect among *compliers* (those moved by the instrument).

## Simulated example: email encouragement for feature adoption

In [ ]:
n = 10000
ability = rng.normal(0, 1, n)            # UNOBSERVED confounder
Z = rng.binomial(1, 0.5, n)              # randomized encouragement email
# adoption driven by unobserved ability AND the email
p_adopt = 1 / (1 + np.exp(-(-0.8 + 1.2*ability + 1.1*Z)))
D = rng.binomial(1, p_adopt)
TRUE_LATE = 15.0
y = 50 + TRUE_LATE*D + 12*ability + rng.normal(0, 10, n)  # ability also raises y
iv = pd.DataFrame(dict(Z=Z, D=D, y=y))

ols_biased = smf.ols("y ~ D", data=iv).fit()
print(f"Naive OLS: {ols_biased.params['D']:.1f}  (true LATE = {TRUE_LATE})")

OLS is badly inflated: high-ability users both adopt more and spend more.

## Two-stage least squares with `linearmodels`

In [ ]:
from linearmodels.iv import IV2SLS

iv_res = IV2SLS.from_formula("y ~ 1 + [D ~ Z]", data=iv).fit(
    cov_type="robust")
print(iv_res.summary.tables[1])
print(f"\nFirst-stage F: {iv_res.first_stage.diagnostics['f.stat'].iloc[0]:.1f}")

The Wald/2SLS estimate recovers ≈ 15. Equivalent intuition: $\hat{\tau}_{LATE} = \frac{\text{ITT on } y}{\text{ITT on } D}$ — the effect of the email on revenue divided by the effect of the email on adoption.

## Practical guardrails

- **Weak instruments** (first-stage F below ~10, or the stricter Lee et al. 2022 threshold of ~104 for standard 2SLS inference) produce biased, wildly uncertain estimates. Report the first stage prominently; with borderline strength use Anderson–Rubin confidence intervals (`linearmodels` provides them).
- **Exclusion violations are the usual killer.** Could the encouragement email itself (not the feature) change behavior — e.g., simply reminding users the product exists? Design instruments where the only plausible channel is treatment.
- **LATE ≠ ATE.** Compliers may be unusual (marginal adopters). Be explicit with stakeholders: "this is the effect for customers whose adoption the email tipped."
- IV + panel data combine naturally: `IV2SLS` accepts exogenous controls (`y ~ 1 + controls + [D ~ Z]`).